# Main questions

<!-- hide-toc -->
## Key Highlights

- **Prestige adoption:** 8,327 users (5.2%) reached Prestige in Diorama, with a further 1,759 reaching tiers 3–6+, for a total of 10,086 prestige-reached users out of 158,753 participants.
- **Revenue concentration:** Completers are 6.3% of Diorama users but account for 15.4% of IAP revenue once they become completers (~2.4× their user share). This share has grown consistently across all four events (8.4% → 9.1% → 14.7% → 15.4%), with Diorama/Prestige driving more revenue from fewer completers.
- **Retention strength:** Completer D1 return rate was 98.9% and D28 97.1% during their event, with a post-event hangover of only 1%–2% relative drop — broadly consistent across all seasons.
- **Behaviour shift from Prestige:** Diorama completers show no measurable hangover so far, suggesting Prestige is driving earlier completion and shifting revenue earlier into the event. A full post-event comparison is pending.
- **Gem economy:** Completers hold ~3× more gems than the average player (ending balance 1,422 vs 431), with steady, gradual growth with each season of Timed Albums. The completers are the main cause of increased gems ending balances as it is shown by the non-completers metrics and percentile differences.


<!-- hide-toc -->

### Q1: General numbers on engagement — how many reached Prestige? Was there an increase?

- **8,327 users** reached Prestige (tier 2) in Diorama — **5.2%** of the 158,753 total Diorama participants. 
- A further 1,759 users went deeper into prestige tiers: tier 3 (1,097 · 0.7%), tier 4 (383 · 0.2%), tier 5 (160 · 0.1%), tier 6+ (119).
- For context, completers (proxy for prestigers) grew strongly event over event before Diorama: JoesTastyTravels **2,381** (2.6%) → WinterTales **7,479** (4.1%, +214%) → AppletonLove **12,559** (7.5%, +68%) → Diorama **9,390** (6.3%, −25%).
- The dip in Diorama completers relative to AppletonLove is partly explained by better game balance and partly by the inflated completer count in AppletonLove driven by OrdersV3.

([Completers > Per Event > Users](#Users)) ([Prestigers > Engagement](#Engagement)) 

### Q2: Did we see a revenue uplift for completers? What is the actual % improvement? How do completers perform vs previous timed album instances?

- Completers are **6.3% of Diorama users** but generate **15.4% of IAP revenue** once they become completers (~2.4× their user share) and **5.1% of ad revenue**.
- The IAP contribution of completers has grown consistently across events: 8.4% → 9.1% → 14.7% → **15.4%** — the feature is increasingly self-selecting high-value payers, and Prestige appears to be driving more revenue from fewer completers. NOTE: if we consider completers users total revenue contribution (even before becoming a completer) they can add up to 70% of total IAP contribution, confirming these are mainly payers and have a high incidence on revenue.
- Ad revenue share also grew event over event but appears to have peaked at AppletonLove: 2.0% → 3.1% → 5.2% → **5.1%** (flat vs AppletonLove). NOTE: Ad revenue has spiked afer Diorama for completers. It is worth investigating if this is driven by having seen prestige or something else.

([Completers > Full Event > Revenue](#Revenue)) ([Completers > Daily > Revenue](#Revenue))

### Q3: Hangover for completers — is there one and how does it compare to previous runs?

- The hangover effect exists and is visible in a couple of metrics:
- Return rate D1: the engagement hangover in completers is noticeable, but the drop is very small and, because they are highly engaged players, the return rate remains very healthy. There is no significant difference in hangover between seasons; the relative drop is between 1%–2%.
- ARPU also shows a slight hangover for completers, but the impact has been improving (decreasing) with each season, from −12.91% on Joe's Tasty Travels to −6.24% on AppletonLove.
- Prestige in Diorama has definitely caused a change in completer behaviour, as no hangover is visible so far; instead there appears to be a push for early completion visible in ARPU metrics. This should be interpreted with caution, as Diorama completers have not yet had a full subsequent Timed Album season for a fair comparison.

([Completers > Daily > Hangover > return-rate-engagement](#return-rate-engagement)) ([Completers > Daily > Hangover > arpu](#arpu))


### Q4: Gem inventories — are completers inflating their balances?

- The rising gem inventory is definitely driven by completers. The breakdown by completers/non-completers and the percentile charts show that the average increase is driven by them.

([Economy > Gems > Ending balances](#ending-balances)) ([Ending balances percentiles](#ending-balances-percentiles))

### PENDING — Product categories

- More detailed analysis on what completers/prestigers are spending on


In [2]:
# hide-output
import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.graph_objects as go
import plotly.express as px
import numpy as np


<!-- hide --> 
## Aux functions

Helper functions used throughout the notebook. Defines the vertical line events config and the `add_vlines_to_figure` utility for annotating charts with key liveops dates.

In [3]:
# Define vlines configuration
vlines_events = [
    {'x': pd.Timestamp('2025-11-04').timestamp() * 1000, 'annotation_text': 'JoesTastyTravels'},
    {'x': pd.Timestamp('2025-12-16').timestamp() * 1000, 'annotation_text': 'WinterTales'},
    {'x': pd.Timestamp('2026-01-26').timestamp() * 1000, 'annotation_text': 'AppletonLove'},
    {'x': pd.Timestamp('2026-03-08').timestamp() * 1000, 'annotation_text': 'Diorama'},
    {'x': pd.Timestamp('2026-04-18').timestamp() * 1000, 'annotation_text': 'ThroughTheAges'},
]

def add_vlines_to_figure(fig, vlines_config):
    """
    Add multiple vertical lines to a plotly figure.
    
    Args:
        fig: plotly figure object
        vlines_config: list of dicts with keys 'x', 'annotation_text', and optional 'line_dash', 'line_color', 'line_width'
    """
    for vline in vlines_config:
        fig.add_vline(
            x=vline['x'],
            line_dash=vline.get('line_dash', 'dash'),
            line_color=vline.get('line_color', 'gray'),
            line_width=vline.get('line_width', 1),
            annotation_text=vline['annotation_text'],
            annotation_position='top right'
        )
    return fig


<!-- hide --> 
# Get data

Loads all raw data from BigQuery and caches locally as pickles for fast iteration.

<!-- hide --> 
## activity

Pulls user-level daily activity for each Timed Album event, including sticker packs opened, sets completed, tier reached, and IAP/ad revenue.

In [4]:
# hide-output
query_location = './sql/prestige.sql'
parameters = {
    'start_date':'2025-11-01',
}


bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 64.11 GB when run.
Estimated query cost: $0.43


In [5]:
refresh_flag = True

data = pd.DataFrame()
if refresh_flag:
    data = bqc.get(query='./sql/prestige.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/prestige_data.pkl')
else:
    data = pd.read_pickle('./data/prestige_data.pkl')

In [6]:
# hide-output
data.columns

Index(['user_id', 'dt', 'max_level', 'liveops_event_definition_id',
       'liveops_iteration_id', 'liveops_iteration_counter',
       'liveops_user_start_dt', 'days_since_event_start', 'active',
       'n_sessions', 'n_stickerpack_opened', 'n_stickerpack_opened_cumu',
       'n_stickers_collected', 'n_stickers_collected_cumu',
       'n_new_stickers_collected', 'n_new_stickers_collected_cumu',
       'n_dupe_stickers_collected', 'n_dupe_stickers_collected_cumu',
       'n_dreamsticker_used', 'n_dreamsticker_used_cumu', 'dupepoints_inflow',
       'dupepoints_inflow_cumu', 'dupepoints_outflow',
       'dupepoints_outflow_cumu', 'dupepoints_balance_start',
       'dupepoints_balance_end', 'n_sets_completed', 'max_tier_reached',
       'loading_timestamp', 'gross_usd_iap_revenue', 'net_usd_iap_revenue',
       'gross_usd_ad_revenue', 'net_usd_ad_revenue', 'user_id_d1',
       'user_id_d3', 'user_id_d7', 'user_id_d14', 'user_id_d28'],
      dtype='object')

<!-- hide --> 
## Economy flows

Pulls gem and economy flow data for deeper inventory and spending analysis.

In [7]:
# hide-output
query_location = './sql/economyflow.sql'
parameters = {
    'start_date':'2025-11-01',
}


bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 431.33 GB when run.
Estimated query cost: $2.89


In [8]:
refresh_flag = True
data_econ = pd.DataFrame()
if refresh_flag:
    data_econ = bqc.get(query='./sql/economyflow.sql', is_path=True, query_parameters=parameters)
    data_econ.to_pickle('./data/economyflow.pkl')
else:
    data_econ = pd.read_pickle('./data/economyflow.pkl')

In [75]:
# hide-output
data_econ.columns

Index(['dt', 'user_id', 'currency', 'balance_start', 'balance_end',
       'n_economy_inflow', 'n_economy_outflow', 'completer_any_flag'],
      dtype='object')

<!-- hide --> 
## Fix some data

Merges the `S5_ThroughTheAges` duplicate into `ThroughTheAges`, fills missing event IDs as `NoTimedAlbum`, and caps `days_since_event_start` at 39.

In [76]:
# hide-output

min_dt_per_event = data.groupby('liveops_event_definition_id')['dt'].min().reset_index()
min_dt_per_event.columns = ['liveops_event_definition_id', 'min_dt']
min_dt_per_event

,liveops_event_definition_id,min_dt
0,NoTimedAlbum,2025-11-01
1,TimedAlbum-AppletonLove,2026-01-25
2,TimedAlbum-Diorama,2026-03-07
3,TimedAlbum-JoesTastyTravels,2025-11-03
4,TimedAlbum-ThroughTheAges,2026-04-17
5,TimedAlbum-WinterTales,2025-12-15


In [77]:
# hide-output
data.liveops_event_definition_id.unique()

array(['TimedAlbum-ThroughTheAges', 'TimedAlbum-JoesTastyTravels',
       'TimedAlbum-WinterTales', 'TimedAlbum-AppletonLove',
       'TimedAlbum-Diorama', 'NoTimedAlbum'], dtype=object)

In [78]:
# hide-output
data['liveops_event_definition_id'] = data['liveops_event_definition_id'].replace('TimedAlbum-S5_ThroughTheAges', 'TimedAlbum-ThroughTheAges')
data['liveops_event_definition_id'] = data['liveops_event_definition_id'].fillna('NoTimedAlbum')
data.liveops_event_definition_id.unique()

array(['TimedAlbum-ThroughTheAges', 'TimedAlbum-JoesTastyTravels',
       'TimedAlbum-WinterTales', 'TimedAlbum-AppletonLove',
       'TimedAlbum-Diorama', 'NoTimedAlbum'], dtype=object)

In [79]:
# hide-output
data['days_since_event_start'] = data['days_since_event_start'].apply(lambda x: x if x <= 39 else None)
data['days_since_event_start'].unique()

array([22., 19., 14., 10., 33., 32., 18., 30., nan, 23., 13., 26., 21.,
       29., 38., 24., 25., 27., 34., 36., 16., 20.,  8., 15., 31., 39.,
       11., 12., 28., 37.,  6., 35., 17.,  5.,  4.,  7.,  3.,  9.,  2.,
        0.,  1.])

<!-- hide --> 
# Health check and data setup

Creates event-specific dataframes and adds helper columns (`reached_prestige_user_id`, `completer_user_id`, revenue flags) used for all downstream aggregations.

<!-- hide -->
### Daily active users check

Overall daily unique users across all events combined and broken down by event — sanity check on data coverage and event overlap.

In [80]:
# hide-output
custom_order = ['NoTimedAlbum', 'TimedAlbum-JoesTastyTravels', 'TimedAlbum-WinterTales', 'TimedAlbum-AppletonLove', 'TimedAlbum-Diorama',  'TimedAlbum-ThroughTheAges']

In [81]:
# Create a plotly chart showing daily unique users by event
daily_users = data.groupby(['dt'])['user_id'].nunique().reset_index()
daily_users.columns = ['dt', 'unique_users']

daily_users_event = data.groupby(['dt', 'liveops_event_definition_id'])['user_id'].nunique().reset_index()
daily_users_event.columns = ['dt', 'liveops_event_definition_id', 'unique_users']

In [82]:
# hide-output
fig = px.line(daily_users, 
              x='dt', 
              y='unique_users',
              title='Daily Unique Users by Event',
              width=1200,
              height=600,
              hover_data={'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.update_yaxes(rangemode='tozero')
fig.show()

In [83]:
# hide-output
fig = px.line(daily_users_event, 
              x='dt', 
              y='unique_users',
              color='liveops_event_definition_id',
              title='Daily Unique Users by Event',
              width=1200,
              height=600,
              hover_data={'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.update_yaxes(rangemode='tozero')
fig.show()

<!-- hide -->
### Define prestigers

In [84]:
# hide-output
users_reached_prestige = data[(data['liveops_event_definition_id'] == 'TimedAlbum-Diorama') & (data['max_tier_reached'] == 2)]['user_id'].drop_duplicates()
users_reached_prestige

1135        8C9E76B86F1888B3
4331        DB28CC2A74F6A7CF
5460        CA4C7CF8CA176918
6411        FF3002075DC97560
7201        4F7B97A294827C74
                  ...       
19509895    D3D05B2A32BD3B00
19795517    553EE71A688B027E
21468873    45E2419654A74571
21634362    9D1DB3F6A6795E9D
21693178    63778B2ED7A02A00
Name: user_id, Length: 8330, dtype: object

<!-- hide -->
### Extend data  and define completers

In [85]:
# hide-output

data_extended = data.copy()
data_extended = data_extended.merge(
    users_reached_prestige.to_frame().rename(columns={'user_id': 'reached_prestige_user_id'}),
    left_on='user_id',
    right_on='reached_prestige_user_id',
    how='left'
)

# completers
level_reached_completer = 12
data_extended['completer_user_id'] = data_extended['user_id'].where(data_extended['n_sets_completed'] == level_reached_completer, None)
data_extended['iap_revenue_completers'] = data_extended['gross_usd_iap_revenue'].where(data_extended['n_sets_completed'] == level_reached_completer, 0)
data_extended['ad_revenue_completers'] = data_extended['gross_usd_ad_revenue'].where(data_extended['n_sets_completed'] == level_reached_completer, 0)

users_completer_JoesTastyTravels = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-JoesTastyTravels') & (data_extended['n_sets_completed'] == level_reached_completer)].user_id.drop_duplicates()
users_completer_WinterTales = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-WinterTales') & (data_extended['n_sets_completed'] == level_reached_completer)].user_id.drop_duplicates()
users_completer_AppletonLove = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-AppletonLove') & (data_extended['n_sets_completed'] == level_reached_completer)].user_id.drop_duplicates()
users_completer_Diorama = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-Diorama') & (data_extended['n_sets_completed'] == level_reached_completer)].user_id.drop_duplicates()
users_completer_ThroughTheAges = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-ThroughTheAges') & (data_extended['n_sets_completed'] == level_reached_completer)].user_id.drop_duplicates()

users_completers_any = pd.concat([
    users_completer_JoesTastyTravels,
    users_completer_WinterTales,
    users_completer_AppletonLove,
    users_completer_Diorama,
    users_completer_ThroughTheAges
]).drop_duplicates()

# near completers
level_reached_near_completer = 10
data_extended['near_completer_user_id'] = data_extended['user_id'].where(data_extended['n_sets_completed'] == level_reached_near_completer, None)
data_extended['iap_revenue_near_completers'] = data_extended['gross_usd_iap_revenue'].where(data_extended['n_sets_completed'] == level_reached_near_completer, 0)
data_extended['ad_revenue_near_completers'] = data_extended['gross_usd_ad_revenue'].where(data_extended['n_sets_completed'] == level_reached_near_completer, 0)

users_near_completer_JoesTastyTravels = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-JoesTastyTravels') & (data_extended['n_sets_completed'] == level_reached_near_completer)].user_id.drop_duplicates()
users_near_completer_WinterTales = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-WinterTales') & (data_extended['n_sets_completed'] == level_reached_near_completer)].user_id.drop_duplicates()
users_near_completer_AppletonLove = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-AppletonLove') & (data_extended['n_sets_completed'] == level_reached_near_completer)].user_id.drop_duplicates()
users_near_completer_Diorama = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-Diorama') & (data_extended['n_sets_completed'] == level_reached_near_completer)].user_id.drop_duplicates()
users_near_completer_ThroughTheAges = data_extended[(data_extended['liveops_event_definition_id'] == 'TimedAlbum-ThroughTheAges') & (data_extended['n_sets_completed'] == level_reached_near_completer)].user_id.drop_duplicates()

data_extended

,user_id,dt,max_level,liveops_event_definition_id,liveops_iteration_id,liveops_iteration_counter,liveops_user_start_dt,days_since_event_start,active,n_sessions,...,user_id_d7,user_id_d14,user_id_d28,reached_prestige_user_id,completer_user_id,iap_revenue_completers,ad_revenue_completers,near_completer_user_id,iap_revenue_near_completers,ad_revenue_near_completers
0,C863A37737202455,2026-05-09,18,TimedAlbum-ThroughTheAges,202604171000,1,2026-05-04,22.0,1,1,...,C863A37737202455,None,None,NaN,None,0.0,0.0,None,0.0,0.0
1,C4012B78D1E33B7B,2026-05-06,37,TimedAlbum-ThroughTheAges,202604171000,4,2026-04-20,19.0,1,1,...,C4012B78D1E33B7B,C4012B78D1E33B7B,None,NaN,None,0.0,0.0,None,0.0,0.0
2,F487D5DBF63BAF72,2025-11-17,143,TimedAlbum-JoesTastyTravels,202511031000,1,2025-11-06,14.0,1,1,...,F487D5DBF63BAF72,F487D5DBF63BAF72,F487D5DBF63BAF72,NaN,None,0.0,0.0,None,0.0,0.0
3,C8F53CBF5F4153BB,2026-04-27,23,TimedAlbum-ThroughTheAges,202604171000,4,2026-04-27,10.0,1,3,...,None,C8F53CBF5F4153BB,None,NaN,None,0.0,0.0,None,0.0,0.0
4,B6095809E928B87A,2026-01-03,22,TimedAlbum-WinterTales,202512151000,1,2025-12-31,19.0,1,2,...,B6095809E928B87A,B6095809E928B87A,None,NaN,None,0.0,0.0,None,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21993396,C023649075E1F229,2026-05-10,140,TimedAlbum-ThroughTheAges,202604171000,4,2026-04-18,23.0,1,3,...,None,None,None,NaN,None,0.0,0.0,None,0.0,0.0
21993397,7DB4679D37F501BF,2026-04-18,91,TimedAlbum-ThroughTheAges,202604171000,5,2026-04-18,1.0,1,1,...,7DB4679D37F501BF,None,None,NaN,None,0.0,0.0,None,0.0,0.0
21993398,6CADC0BEA3B4A316,2026-05-13,25,TimedAlbum-ThroughTheAges,202604171000,4,2026-04-18,26.0,1,1,...,None,None,None,NaN,None,0.0,0.0,None,0.0,0.0
21993399,167520F2EC0960A2,2026-05-01,69,TimedAlbum-ThroughTheAges,202604171000,5,2026-04-18,14.0,1,3,...,None,None,None,NaN,None,0.0,0.0,None,0.0,0.0


In [86]:
# hide-output
data_extended['reached_prestige_user_id'].unique()

array([nan, '9B3D69F587BCB016', 'F1F0BCC96C95B3A4', ...,
       '16E514B5E5269A70', 'FBFAD15FA33F2F61', '80671E931B05B9D'],
      shape=(8331,), dtype=object)

# Completers

Analysis of users who completed all 12 sets in a Timed Album event — how many there are, what share of total players they represent, and how much revenue they drive.

## Full Event

Aggregate view per event: total completers, completion rate, IAP/ad revenue from completers, and their share of total event revenue.

In [87]:
# hide-output
data_completers_per_event = data_extended.groupby(['liveops_event_definition_id']).agg(
    unique_users =('user_id', 'nunique'),
    iap_revenue=('gross_usd_iap_revenue', 'sum'),
    ad_revenue=('gross_usd_ad_revenue', 'sum'),
    unique_completers=('completer_user_id', 'nunique'),
    iap_revenue_completers=('iap_revenue_completers', 'sum'),
    ad_revenue_completers=('ad_revenue_completers', 'sum'),
    ).reset_index()

data_completers_per_event['liveops_event_definition_id'] = pd.Categorical(
    data_completers_per_event['liveops_event_definition_id'], 
    categories=custom_order, 
    ordered=True
)
data_completers_per_event = data_completers_per_event.sort_values('liveops_event_definition_id')

data_completers_per_event['pct_change_completers'] = data_completers_per_event['unique_completers'].pct_change()
data_completers_per_event['pct_change_iap_revenue_completers'] = data_completers_per_event['iap_revenue_completers'].pct_change()
data_completers_per_event['pct_change_adrevenue_completers'] = data_completers_per_event['ad_revenue_completers'].pct_change()

data_completers_per_event['percentage_completers'] = data_completers_per_event['unique_completers'] / data_completers_per_event['unique_users']
data_completers_per_event['percentage_iap_revenue_completers'] = data_completers_per_event['iap_revenue_completers'] / data_completers_per_event['iap_revenue']
data_completers_per_event['percentage_ad_revenue_completers'] = data_completers_per_event['ad_revenue_completers'] / data_completers_per_event['ad_revenue']

data_completers_per_event

,liveops_event_definition_id,unique_users,iap_revenue,ad_revenue,unique_completers,iap_revenue_completers,ad_revenue_completers,pct_change_completers,pct_change_iap_revenue_completers,pct_change_adrevenue_completers,percentage_completers,percentage_iap_revenue_completers,percentage_ad_revenue_completers
0,NoTimedAlbum,655171,1.095246e+06,356459.244821,0,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
3,TimedAlbum-JoesTastyTravels,91854,8.745843e+05,229000.364251,2381,73833.232535,4555.938313,inf,inf,inf,0.025922,0.084421,0.019895
5,TimedAlbum-WinterTales,182398,1.598864e+06,445696.113511,7479,144867.002993,13972.668547,2.141117,0.962084,2.066913,0.041004,0.090606,0.031350
1,TimedAlbum-AppletonLove,168225,1.683723e+06,443527.449654,12559,247075.219796,22843.074811,0.679235,0.705531,0.634840,0.074656,0.146743,0.051503
2,TimedAlbum-Diorama,146303,1.405380e+06,409574.534144,9390,216540.096002,20935.724527,-0.252329,-0.123586,-0.083498,0.064182,0.154079,0.051116
4,TimedAlbum-ThroughTheAges,129164,1.036807e+06,312837.144408,2236,66562.299637,4380.358365,-0.761874,-0.692610,-0.790771,0.017311,0.064199,0.014002


### Users

Bar charts of absolute completer counts and completion rate (completers / total users) per event.

In [88]:
fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='unique_completers',
              color='liveops_event_definition_id',
              title='Unique Completers by Event',
              width=1200,
              height=600,
              hover_data={'unique_completers': True, 'pct_change_completers': ':.2%'},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='percentage_completers',
              color='liveops_event_definition_id',
              title='Completion Rate by Event (Completers / Total Users)',
              width=1200,
              height=600,
              hover_data={'percentage_completers': ':.2%', 'unique_completers': True, 'pct_change_completers': ':.2%'},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()


### Revenue

Bar charts of absolute and percentage IAP and ad revenue contributed by completers, compared across events.

In [89]:
fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='iap_revenue_completers',
              color='liveops_event_definition_id',
              title='Absolute IAP Revenue by completers by Event',
              width=1200,
              height=600,
              hover_data={'iap_revenue_completers': ':.0f', 'unique_completers': True, 'pct_change_iap_revenue_completers': ':.2%'},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='ad_revenue_completers',
              color='liveops_event_definition_id',
              title='Absolute AD Revenue by completers by Event',
              width=1200,
              height=600,
              hover_data={'ad_revenue_completers': ':.0f', 'unique_completers': True, 'pct_change_adrevenue_completers': ':.2%'},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

In [90]:
fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='percentage_iap_revenue_completers',
              color='liveops_event_definition_id',
              title='% IAP Revenue by completers by Event',
              width=1200,
              height=600,
              hover_data={'percentage_iap_revenue_completers': ':.2%', 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

fig = px.bar(data_completers_per_event, 
              x='liveops_event_definition_id', 
              y='percentage_ad_revenue_completers',
              color='liveops_event_definition_id',
              title='% AD Revenue by completers by Event',
              width=1200,
              height=600,
              hover_data={'percentage_ad_revenue_completers': ':.2%', 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

## Daily

Time-series breakdowns of completer activity and revenue, both by calendar date and by days since event start.

In [91]:
# hide-output
data_completers_daily = data_extended.groupby(['dt', 'liveops_event_definition_id']).agg(
    unique_users =('user_id', 'nunique'),
    iap_revenue=('gross_usd_iap_revenue', 'sum'),
    ad_revenue=('gross_usd_ad_revenue', 'sum'),
    unique_completers=('completer_user_id', 'nunique'),
    iap_revenue_completers=('iap_revenue_completers', 'sum'),
    ad_revenue_completers=('ad_revenue_completers', 'sum'),
    unique_usersd1 = ('user_id_d1', 'nunique'),
    unique_usersd7 = ('user_id_d7', 'nunique'),
    unique_usersd14 = ('user_id_d14', 'nunique'),
    unique_usersd28 = ('user_id_d28', 'nunique'),
    ).reset_index()

data_completers_daily['percentage_completers'] = data_completers_daily['unique_completers'] / data_completers_daily['unique_users']

data_completers_daily['return_rate_d1'] = data_completers_daily['unique_usersd1'] / data_completers_daily['unique_users']
data_completers_daily['return_rate_d7'] = data_completers_daily['unique_usersd7'] / data_completers_daily['unique_users']
data_completers_daily['return_rate_d14'] = data_completers_daily['unique_usersd14'] / data_completers_daily['unique_users']
data_completers_daily['return_rate_d28'] = data_completers_daily['unique_usersd28'] / data_completers_daily['unique_users']


data_completers_daily

,dt,liveops_event_definition_id,unique_users,iap_revenue,ad_revenue,unique_completers,iap_revenue_completers,ad_revenue_completers,unique_usersd1,unique_usersd7,unique_usersd14,unique_usersd28,percentage_completers,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28
0,2025-11-01,NoTimedAlbum,120270,40196.046637,11009.299532,0,0.000000,0.000000,99943,90707,85376,77243,0.000000,0.830989,0.754195,0.709869,0.642247
1,2025-11-02,NoTimedAlbum,125102,33973.254226,11371.297612,0,0.000000,0.000000,101748,94556,88933,80477,0.000000,0.813320,0.755831,0.710884,0.643291
2,2025-11-03,NoTimedAlbum,80239,17757.237587,6375.853601,0,0.000000,0.000000,62937,57032,53184,47214,0.000000,0.784369,0.710777,0.662820,0.588417
3,2025-11-03,TimedAlbum-JoesTastyTravels,43977,17555.175245,4821.619927,0,0.000000,0.000000,40161,38117,36376,33279,0.000000,0.913227,0.866749,0.827160,0.756736
4,2025-11-04,NoTimedAlbum,72140,17425.765420,6087.081425,0,0.000000,0.000000,57232,51783,48171,42764,0.000000,0.793346,0.717813,0.667743,0.592792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,2026-05-18,TimedAlbum-ThroughTheAges,75517,26734.033300,9196.035909,1264,7587.262621,672.989287,66202,0,0,0,0.016738,0.876650,0.000000,0.000000,0.000000
403,2026-05-19,NoTimedAlbum,8087,1204.957089,408.944822,0,0.000000,0.000000,4448,0,0,0,0.000000,0.550019,0.000000,0.000000,0.000000
404,2026-05-19,TimedAlbum-ThroughTheAges,75132,23787.351592,9138.850545,1643,8933.161655,849.888314,65959,0,0,0,0.021868,0.877908,0.000000,0.000000,0.000000
405,2026-05-20,NoTimedAlbum,8076,637.529226,407.522233,0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000


<!-- hide --> 
### Users

Daily completer user count and completion rate over time (by calendar date and by days since event start), stacked by event.

In [92]:
fig = px.bar(data_completers_daily, 
              x='dt', 
              y='unique_completers',
              color='liveops_event_definition_id',
              title='Daily Unique Completers by Event',
              width=1200,
              height=600,
              hover_data={'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.bar(data_completers_daily, 
              x='dt', 
              y='percentage_completers',
              color='liveops_event_definition_id',
              title='Daily Percentage of Unique Completers by Event',
              width=1200,
              height=600,
              hover_data={'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [93]:
# hide-output
data_completers_dse = data_extended.groupby(['days_since_event_start', 'liveops_event_definition_id']).agg(
    total_users =('user_id', 'nunique'),
    iap_revenue=('gross_usd_iap_revenue', 'sum'),
    ad_revenue=('gross_usd_ad_revenue', 'sum'),
    unique_completers=('completer_user_id', 'nunique'),
    iap_revenue_completers=('iap_revenue_completers', 'mean'),
    ad_revenue_completers=('ad_revenue_completers', 'mean')
    ).reset_index()

data_completers_dse['percentage_completers'] = data_completers_dse['unique_completers'] / data_completers_dse['total_users']

data_completers_dse

,days_since_event_start,liveops_event_definition_id,total_users,iap_revenue,ad_revenue,unique_completers,iap_revenue_completers,ad_revenue_completers,percentage_completers
0,0.0,TimedAlbum-AppletonLove,87361,32848.389784,10125.143145,0,0.000000,0.000000,0.000000
1,0.0,TimedAlbum-Diorama,73935,44926.431033,9225.661589,0,0.000000,0.000000,0.000000
2,0.0,TimedAlbum-JoesTastyTravels,43977,17555.175245,4821.619927,0,0.000000,0.000000,0.000000
3,0.0,TimedAlbum-ThroughTheAges,68240,21532.328273,7550.753011,0,0.000000,0.000000,0.000000
4,0.0,TimedAlbum-WinterTales,84968,32591.825683,9361.364240,0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
189,38.0,TimedAlbum-WinterTales,99627,35698.013339,10064.931896,6903,0.142327,0.015793,0.069288
190,39.0,TimedAlbum-AppletonLove,91800,27917.464359,9353.027614,12204,0.193022,0.027557,0.132941
191,39.0,TimedAlbum-Diorama,82786,33620.588405,9031.432516,9232,0.287339,0.026604,0.111516
192,39.0,TimedAlbum-JoesTastyTravels,50109,42304.180127,5039.682702,2340,0.337697,0.012782,0.046698


In [94]:
fig = px.line(data_completers_dse, 
              x='days_since_event_start', 
              y='percentage_completers',
              color='liveops_event_definition_id',
              title='Percentage of Unique Completers by Event by Days Since Event Start',
              width=1200,
              height=600,
              hover_data={'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})


fig.show()

### Revenue

Daily IAP and ad revenue from completers over time and by days since event start, per event.

In [95]:
fig = px.bar(data_completers_daily, 
              x='dt', 
              y='iap_revenue_completers',
              color='liveops_event_definition_id',
              title='Daily IAP Revenue by Event for completers',
              width=1200,
              height=600,
              hover_data={'iap_revenue_completers': True, 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.bar(data_completers_daily, 
              x='dt', 
              y='ad_revenue_completers',
              color='liveops_event_definition_id',
              title='Daily AD Revenue by Event for completers',
              width=1200,
              height=600,
              hover_data={'ad_revenue_completers': True, 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [96]:
fig = px.line(data_completers_dse, 
              x='days_since_event_start', 
              y='iap_revenue_completers',
              color='liveops_event_definition_id',
              title='Daily IAP Revenue by Event for completers by Days Since Event Start',
              width=1200,
              height=600,
              hover_data={'iap_revenue_completers': True, 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

fig = px.line(data_completers_dse, 
              x='days_since_event_start', 
              y='ad_revenue_completers',
              color='liveops_event_definition_id',
              title='Daily AD Revenue by Event for completers by Days Since Event Start',
              width=1200,
              height=600,
              hover_data={'ad_revenue_completers': True, 'unique_completers': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig.show()

## Hangover

Tracks completer cohorts from each event across all subsequent dates — measures ongoing engagement and ARPDAU after the event ends to detect post-event revenue effects.

In [ ]:
# hide-output
# Create a loop to iterate through all events and aggregate results
events = custom_order
user_sets = [users_completer_JoesTastyTravels, users_completer_WinterTales, users_completer_AppletonLove, users_completer_Diorama, users_completer_ThroughTheAges]
#user_sets = [users_near_completer_JoesTastyTravels, users_near_completer_WinterTales, users_near_completer_AppletonLove, users_near_completer_Diorama, users_near_completer_ThroughTheAges]
user_set_names = ['JoesTastyTravels', 'WinterTales', 'AppletonLove', 'Diorama', 'ThroughTheAges']
results_completers_list = []
results_non_completers_list = []

for set, set_name in zip(user_sets, user_set_names):
    event_completers_data = data_extended[(data_extended['user_id'].isin(set))]
    
    event_completers_aggregated = event_completers_data.groupby(['dt']).agg(
        unique_users=('user_id', 'nunique'),
        iap_revenue=('gross_usd_iap_revenue', 'sum'),
        ad_revenue=('gross_usd_ad_revenue', 'sum'),
        unique_usersd1 = ('user_id_d1', 'nunique'),
        unique_usersd7 = ('user_id_d7', 'nunique'),
        unique_usersd14 = ('user_id_d14', 'nunique'),
        unique_usersd28 = ('user_id_d28', 'nunique'),
    ).reset_index()
    
    event_completers_aggregated['set_completed'] = set_name

    event_completers_aggregated['return_rate_d1'] = event_completers_aggregated['unique_usersd1'] / event_completers_aggregated['unique_users']
    event_completers_aggregated['return_rate_d7'] = event_completers_aggregated['unique_usersd7'] / event_completers_aggregated['unique_users']
    event_completers_aggregated['return_rate_d14'] = event_completers_aggregated['unique_usersd14'] / event_completers_aggregated['unique_users']
    event_completers_aggregated['return_rate_d28'] = event_completers_aggregated['unique_usersd28'] / event_completers_aggregated['unique_users']

    # if return rates equal zero then value is null
    event_completers_aggregated['return_rate_d1'] = event_completers_aggregated['return_rate_d1'].replace(0, None)
    event_completers_aggregated['return_rate_d7'] = event_completers_aggregated['return_rate_d7'].replace(0, None)
    event_completers_aggregated['return_rate_d14'] = event_completers_aggregated['return_rate_d14'].replace(0, None)
    event_completers_aggregated['return_rate_d28'] = event_completers_aggregated['return_rate_d28'].replace(0, None)

    results_completers_list.append(event_completers_aggregated)

data_completers_all_events = pd.concat(results_completers_list, ignore_index=True)

data_completers_all_events['arpdau_completers'] = data_completers_all_events['iap_revenue'] / data_completers_all_events['unique_users']

filter_mask = ['JoesTastyTravels', 'WinterTales', 'AppletonLove', 'Diorama', 'ThroughTheAges']

data_completers_all_events = data_completers_all_events[data_completers_all_events['set_completed'].isin(filter_mask)]

data_completers_all_events

,dt,unique_users,iap_revenue,ad_revenue,unique_usersd1,unique_usersd7,unique_usersd14,unique_usersd28,set_completed,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau_completers
0,2025-11-01,2105,7173.314874,626.561418,2067,2070,2073,2065,JoesTastyTravels,0.981948,0.983373,0.984798,0.980998,3.407751
1,2025-11-02,2123,6013.963992,631.410488,2091,2094,2092,2091,JoesTastyTravels,0.984927,0.98634,0.985398,0.984927,2.832767
2,2025-11-03,2140,7773.309696,687.915881,2114,2117,2108,2117,JoesTastyTravels,0.98785,0.989252,0.985047,0.989252,3.632388
3,2025-11-04,2151,8203.785549,654.473851,2130,2130,2119,2135,JoesTastyTravels,0.990237,0.990237,0.985123,0.992562,3.813940
4,2025-11-05,2162,8442.740820,638.434900,2146,2142,2134,2145,JoesTastyTravels,0.992599,0.990749,0.987049,0.992137,3.905061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,2026-05-16,2213,13377.946266,1043.777348,2203,0,0,0,ThroughTheAges,0.995481,None,None,None,6.045163
1001,2026-05-17,2218,11918.976840,1045.575726,2209,0,0,0,ThroughTheAges,0.995942,None,None,None,5.373750
1002,2026-05-18,2219,10384.954421,1000.270309,2214,0,0,0,ThroughTheAges,0.997747,None,None,None,4.680016
1003,2026-05-19,2225,9917.137222,1048.976407,2206,0,0,0,ThroughTheAges,0.991461,None,None,None,4.457140


In [98]:
# hide-output

# create a lookup dataframe that contains the all dates in the calendar and their days_from_event_start, removing null ones
calendar_lookup = data_extended[['liveops_event_definition_id','dt', 'days_since_event_start']].dropna().drop_duplicates().sort_values('dt').reset_index(drop=True)

calendar_lookup.rename(columns={'liveops_event_definition_id': 'set_running'}, inplace=True)

# get only the string part after the dash in set_running to match with the 'set_completed' column in data_completers_all_events
calendar_lookup['set_running'] = calendar_lookup['set_running'].apply(lambda x: x.split('-', 1)[1] if '-' in x else x)

# merge calendar_lookup with data_completers_all_events to get the days_since_event_start for each date, keeping all rows in data_completers_all_events
data_completers_all_events = data_completers_all_events.merge(calendar_lookup, on='dt', how='left')

data_completers_all_events[['dt', 'set_completed', 'set_running', 'days_since_event_start']].sort_values('dt')

,dt,set_completed,set_running,days_since_event_start
0,2025-11-01,JoesTastyTravels,NaN,NaN
402,2025-11-01,AppletonLove,NaN,NaN
603,2025-11-01,Diorama,NaN,NaN
804,2025-11-01,ThroughTheAges,NaN,NaN
201,2025-11-01,WinterTales,NaN,NaN
...,...,...,...,...
602,2026-05-20,AppletonLove,ThroughTheAges,33.0
401,2026-05-20,WinterTales,ThroughTheAges,33.0
200,2026-05-20,JoesTastyTravels,ThroughTheAges,33.0
803,2026-05-20,Diorama,ThroughTheAges,33.0


In [99]:
# hide-output
fig = px.line(data_completers_all_events, 
              x='dt', 
              y='unique_users',
              color='set_completed',
              title='Daily Unique Users that were completers of each event',
              width=1200,
              height=600,
              hover_data={'iap_revenue': True, 'unique_users': True},
              category_orders={'set': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

#### Return rate (Engagement)

In [100]:
fig = px.line(data_completers_all_events, 
              x='dt', 
              y='return_rate_d1',
              color='set_completed',
              title='Daily Return Rate (D1) by completers of each event',
              width=1200,
              height=600,
              hover_data={'unique_usersd1': True, 'unique_users': True},
              category_orders={'set': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [101]:
# hide-output
data_completers_all_events['set_completed+running'] = data_completers_all_events['set_completed'] + '-' + data_completers_all_events['set_running']
data_completers_all_events

,dt,unique_users,iap_revenue,ad_revenue,unique_usersd1,unique_usersd7,unique_usersd14,unique_usersd28,set_completed,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau_completers,set_running,days_since_event_start,set_completed+running
0,2025-11-01,2105,7173.314874,626.561418,2067,2070,2073,2065,JoesTastyTravels,0.981948,0.983373,0.984798,0.980998,3.407751,NaN,NaN,NaN
1,2025-11-02,2123,6013.963992,631.410488,2091,2094,2092,2091,JoesTastyTravels,0.984927,0.98634,0.985398,0.984927,2.832767,NaN,NaN,NaN
2,2025-11-03,2140,7773.309696,687.915881,2114,2117,2108,2117,JoesTastyTravels,0.98785,0.989252,0.985047,0.989252,3.632388,JoesTastyTravels,0.0,JoesTastyTravels-JoesTastyTravels
3,2025-11-04,2151,8203.785549,654.473851,2130,2130,2119,2135,JoesTastyTravels,0.990237,0.990237,0.985123,0.992562,3.813940,JoesTastyTravels,1.0,JoesTastyTravels-JoesTastyTravels
4,2025-11-05,2162,8442.740820,638.434900,2146,2142,2134,2145,JoesTastyTravels,0.992599,0.990749,0.987049,0.992137,3.905061,JoesTastyTravels,2.0,JoesTastyTravels-JoesTastyTravels
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,2026-05-16,2213,13377.946266,1043.777348,2203,0,0,0,ThroughTheAges,0.995481,None,None,None,6.045163,ThroughTheAges,29.0,ThroughTheAges-ThroughTheAges
1001,2026-05-17,2218,11918.976840,1045.575726,2209,0,0,0,ThroughTheAges,0.995942,None,None,None,5.373750,ThroughTheAges,30.0,ThroughTheAges-ThroughTheAges
1002,2026-05-18,2219,10384.954421,1000.270309,2214,0,0,0,ThroughTheAges,0.997747,None,None,None,4.680016,ThroughTheAges,31.0,ThroughTheAges-ThroughTheAges
1003,2026-05-19,2225,9917.137222,1048.976407,2206,0,0,0,ThroughTheAges,0.991461,None,None,None,4.457140,ThroughTheAges,32.0,ThroughTheAges-ThroughTheAges


In [102]:
# hide-output

order = ['JoesTastyTravels', 'WinterTales', 'AppletonLove', 'Diorama', 'ThroughTheAges']

data_completers_all_events_aggregated = data_completers_all_events.groupby(['set_completed+running','set_completed','set_running']).agg(
    unique_users=('unique_users', 'sum'),
    iap_revenue=('iap_revenue', 'sum'),
    ad_revenue=('ad_revenue', 'sum'),
    return_rate_d1=('return_rate_d1', 'mean'),
    return_rate_d7=('return_rate_d7', 'mean'),
    return_rate_d14=('return_rate_d14', 'mean'),
    return_rate_d28=('return_rate_d28', 'mean')
    ).reset_index()

data_completers_all_events_aggregated['arpdau'] = data_completers_all_events_aggregated['iap_revenue'] / data_completers_all_events_aggregated['unique_users']

# Sort data_completers_all_events_aggregated by custom order
data_completers_all_events_aggregated['set_completed'] = pd.Categorical(
    data_completers_all_events_aggregated['set_completed'],
    categories=order,
    ordered=True
)
data_completers_all_events_aggregated['set_running'] = pd.Categorical(
    data_completers_all_events_aggregated['set_running'],
    categories=order,
    ordered=True
)

data_completers_all_events_aggregated = data_completers_all_events_aggregated.sort_values(
    by=['set_completed', 'set_running']
).reset_index(drop=True)

data_completers_all_events_aggregated

,set_completed+running,set_completed,set_running,unique_users,iap_revenue,ad_revenue,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau
0,JoesTastyTravels-JoesTastyTravels,JoesTastyTravels,JoesTastyTravels,91148,368641.479890,28880.241639,0.989848,0.984555,0.974755,0.943713,4.044428
1,JoesTastyTravels-WinterTales,JoesTastyTravels,WinterTales,85005,299422.257315,25571.936207,0.977636,0.95116,0.927703,0.891945,3.522408
2,JoesTastyTravels-AppletonLove,JoesTastyTravels,AppletonLove,74955,238420.051353,22797.886206,0.97529,0.951682,0.928838,0.881924,3.180843
3,JoesTastyTravels-Diorama,JoesTastyTravels,Diorama,65410,208236.665506,20520.856828,0.970631,0.949575,0.931155,0.901704,3.183560
4,JoesTastyTravels-ThroughTheAges,JoesTastyTravels,ThroughTheAges,51125,162668.367880,18018.621479,0.967707,0.948357,0.93469,0.902726,3.181777
5,WinterTales-JoesTastyTravels,WinterTales,JoesTastyTravels,243914,507618.062358,56102.474907,0.984291,0.978754,0.976852,0.978008,2.081135
6,WinterTales-WinterTales,WinterTales,WinterTales,287236,718948.902677,73159.589295,0.98957,0.982882,0.974945,0.948222,2.502990
7,WinterTales-AppletonLove,WinterTales,AppletonLove,269756,622759.773175,68342.519251,0.981391,0.954597,0.927602,0.876751,2.308604
8,WinterTales-Diorama,WinterTales,Diorama,230454,495834.025716,57688.207072,0.974645,0.951947,0.930864,0.894848,2.151553
9,WinterTales-ThroughTheAges,WinterTales,ThroughTheAges,176399,368097.591982,48367.286245,0.971747,0.952027,0.934684,0.900845,2.086733


In [103]:
# hide-output

# Find the row where set_completed == set_running for each event
data_completers_all_events_aggregated['sequence'] = 0

for event in order:
    event_mask = data_completers_all_events_aggregated['set_completed'] == event
    event_indices = data_completers_all_events_aggregated[event_mask].index.tolist()
    
    # Find the index where set_completed == set_running
    zero_idx = None
    zero_position = None
    for i, idx in enumerate(event_indices):
        if data_completers_all_events_aggregated.loc[idx, 'set_completed'] == data_completers_all_events_aggregated.loc[idx, 'set_running']:
            zero_idx = idx
            zero_position = i
            break
    
    if zero_idx is not None:
        # Assign sequence numbers: negative before zero, positive after
        for i, idx in enumerate(event_indices):
            data_completers_all_events_aggregated.loc[idx, 'sequence'] = i - zero_position

data_completers_all_events_aggregated


,set_completed+running,set_completed,set_running,unique_users,iap_revenue,ad_revenue,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau,sequence
0,JoesTastyTravels-JoesTastyTravels,JoesTastyTravels,JoesTastyTravels,91148,368641.479890,28880.241639,0.989848,0.984555,0.974755,0.943713,4.044428,0
1,JoesTastyTravels-WinterTales,JoesTastyTravels,WinterTales,85005,299422.257315,25571.936207,0.977636,0.95116,0.927703,0.891945,3.522408,1
2,JoesTastyTravels-AppletonLove,JoesTastyTravels,AppletonLove,74955,238420.051353,22797.886206,0.97529,0.951682,0.928838,0.881924,3.180843,2
3,JoesTastyTravels-Diorama,JoesTastyTravels,Diorama,65410,208236.665506,20520.856828,0.970631,0.949575,0.931155,0.901704,3.183560,3
4,JoesTastyTravels-ThroughTheAges,JoesTastyTravels,ThroughTheAges,51125,162668.367880,18018.621479,0.967707,0.948357,0.93469,0.902726,3.181777,4
5,WinterTales-JoesTastyTravels,WinterTales,JoesTastyTravels,243914,507618.062358,56102.474907,0.984291,0.978754,0.976852,0.978008,2.081135,-1
6,WinterTales-WinterTales,WinterTales,WinterTales,287236,718948.902677,73159.589295,0.98957,0.982882,0.974945,0.948222,2.502990,0
7,WinterTales-AppletonLove,WinterTales,AppletonLove,269756,622759.773175,68342.519251,0.981391,0.954597,0.927602,0.876751,2.308604,1
8,WinterTales-Diorama,WinterTales,Diorama,230454,495834.025716,57688.207072,0.974645,0.951947,0.930864,0.894848,2.151553,2
9,WinterTales-ThroughTheAges,WinterTales,ThroughTheAges,176399,368097.591982,48367.286245,0.971747,0.952027,0.934684,0.900845,2.086733,3


In [104]:
fig = px.line(data_completers_all_events, 
              x='days_since_event_start', 
              y='return_rate_d1',
              color='set_completed+running',
              title='Daily Return Rate (D1) by completers of each event',
              width=1200,
              height=600,
              hover_data={'unique_usersd1': True, 'unique_users': True},
              category_orders={'set': custom_order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(data_completers_all_events, 
              x='days_since_event_start', 
              y='return_rate_d7',
              color='set_completed+running',
              title='Daily Return Rate (D7) by completers of each event',
              width=1200,
              height=600,
              hover_data={'unique_usersd7': True, 'unique_users': True},
              category_orders={'set': custom_order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [105]:
fig = px.box(data_completers_all_events, 
              x='set_running', 
              y='return_rate_d1',
              color='set_completed',
              title='Daily Return Rate (D1) by completers of each event',
              width=1200,
              height=600,
              hover_data={'return_rate_d1': True, 'unique_users': True},
              category_orders={'set': order,'set_on_date': order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [106]:
# Convert return_rate columns from object to numeric
data_completers_all_events_aggregated['return_rate_d1'] = pd.to_numeric(data_completers_all_events_aggregated['return_rate_d1'], errors='coerce')

fig = px.bar(data_completers_all_events_aggregated, 
              x='set_running', 
              y='return_rate_d1',
              color='set_completed',
              barmode='group',
              title='D1 Return Rate by Completers of Each Event',
              width=1200,
              height=600,
              hover_data={'return_rate_d1': ':2%'},
              category_orders={'set_completed': order, 'set_running': order},
              text='return_rate_d1')

fig.update_traces(textposition='outside', texttemplate='%{text:.2%}')

# Set y-axis to start from a value close to the minimum instead of zero
min_val = data_completers_all_events_aggregated['return_rate_d1'].min()
max_val = data_completers_all_events_aggregated['return_rate_d1'].max()
y_range = max_val - min_val
padding = y_range * 0.1

fig.update_yaxes(tickformat='.2%', range=[max(0, min_val - padding), max_val + padding])

fig.show()

In [107]:
# hide-output
# Initialize the column if it doesn't exist
if 'return_rate_d1_relative_diff' not in data_completers_all_events_aggregated.columns:
    data_completers_all_events_aggregated['return_rate_d1_relative_diff'] = np.nan

# Calculate return_rate_d1_relative_diff for all events
for event in order:
    event_data = data_completers_all_events_aggregated[
        data_completers_all_events_aggregated['set_completed'] == event
    ].copy()
    
    
    # Get the baseline value when set_running == set_completed
    baseline = event_data[event_data['set_running'] == event]['return_rate_d1'].values
    
    if len(baseline) > 0:
        # Calculate relative diff for all rows in this event
        event_data['return_rate_d1_relative_diff'] = (event_data['return_rate_d1'] - baseline[0]) / baseline[0]
        
        # Update the main dataframe
        data_completers_all_events_aggregated.loc[
            data_completers_all_events_aggregated['set_completed'] == event,
            'return_rate_d1_relative_diff'
        ] = event_data['return_rate_d1_relative_diff'].values


data_completers_all_events_aggregated

,set_completed+running,set_completed,set_running,unique_users,iap_revenue,ad_revenue,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau,sequence,return_rate_d1_relative_diff
0,JoesTastyTravels-JoesTastyTravels,JoesTastyTravels,JoesTastyTravels,91148,368641.479890,28880.241639,0.989848,0.984555,0.974755,0.943713,4.044428,0,0.000000
1,JoesTastyTravels-WinterTales,JoesTastyTravels,WinterTales,85005,299422.257315,25571.936207,0.977636,0.95116,0.927703,0.891945,3.522408,1,-0.012338
2,JoesTastyTravels-AppletonLove,JoesTastyTravels,AppletonLove,74955,238420.051353,22797.886206,0.975290,0.951682,0.928838,0.881924,3.180843,2,-0.014707
3,JoesTastyTravels-Diorama,JoesTastyTravels,Diorama,65410,208236.665506,20520.856828,0.970631,0.949575,0.931155,0.901704,3.183560,3,-0.019414
4,JoesTastyTravels-ThroughTheAges,JoesTastyTravels,ThroughTheAges,51125,162668.367880,18018.621479,0.967707,0.948357,0.93469,0.902726,3.181777,4,-0.022368
5,WinterTales-JoesTastyTravels,WinterTales,JoesTastyTravels,243914,507618.062358,56102.474907,0.984291,0.978754,0.976852,0.978008,2.081135,-1,-0.005335
6,WinterTales-WinterTales,WinterTales,WinterTales,287236,718948.902677,73159.589295,0.989570,0.982882,0.974945,0.948222,2.502990,0,0.000000
7,WinterTales-AppletonLove,WinterTales,AppletonLove,269756,622759.773175,68342.519251,0.981391,0.954597,0.927602,0.876751,2.308604,1,-0.008265
8,WinterTales-Diorama,WinterTales,Diorama,230454,495834.025716,57688.207072,0.974645,0.951947,0.930864,0.894848,2.151553,2,-0.015082
9,WinterTales-ThroughTheAges,WinterTales,ThroughTheAges,176399,368097.591982,48367.286245,0.971747,0.952027,0.934684,0.900845,2.086733,3,-0.018011


In [108]:
fig = px.line(data_completers_all_events_aggregated, 
              x='set_running', 
              y='return_rate_d1_relative_diff',
              color='set_completed',
              title='Relative Difference in Return Rate by Completers of Each Event (%)',
              width=1200,
              height=600,
              hover_data={'return_rate_d1_relative_diff': ':.2%'},
              category_orders={'set_completed': order, 'set_running': order},
              text='return_rate_d1_relative_diff')

fig.update_traces(textposition='top center', texttemplate='%{text:.2%}')
fig.update_yaxes(title='Return Rate (%)')

fig.show()

fig = px.line(data_completers_all_events_aggregated, 
              x='sequence', 
              y='return_rate_d1_relative_diff',
              color='set_completed',
              title='Relative Difference in Return Rate by Completers of Each Event (%)',
              width=1200,
              height=600,
              hover_data={'return_rate_d1_relative_diff': ':.2%'},
              text='return_rate_d1_relative_diff')

fig.update_traces(textposition='top center', texttemplate='%{text:.4%}%')
fig.update_yaxes(title='Return Rate (%)')

fig.show()

#### Revenue

In [109]:
fig = px.line(data_completers_all_events, 
              x='dt', 
              y='iap_revenue',
              color='set_completed',
              title='Daily IAP Revenue by completers of each event',
              width=1200,
              height=600,
              hover_data={'iap_revenue': True, 'unique_users': True},
              category_orders={'set': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(data_completers_all_events, 
              x='dt', 
              y='ad_revenue',
              color='set_completed',
              title='Daily AD Revenue by completers of each event',
              width=1200,
              height=600,
              hover_data={'ad_revenue': True, 'unique_users': True},
              category_orders={'set': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [110]:
fig = px.line(data_completers_all_events, 
              x='days_since_event_start', 
              y='iap_revenue',
              color='set_completed+running',
              title='Daily IAP Revenue by completers of each event by days since event start',
              width=1200,
              height=600,
              hover_data={'iap_revenue': True, 'unique_users': True},
              category_orders={'set': custom_order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(data_completers_all_events, 
              x='days_since_event_start', 
              y='ad_revenue',
              color='set_completed+running',
              title='Daily Ad Revenue by completers of each event by days since event start',
              width=1200,
              height=600,
              hover_data={'ad_revenue': True, 'unique_users': True},
              category_orders={'set': custom_order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

#### ARPU

In [111]:
fig = px.line(data_completers_all_events, 
              x='dt', 
              y='arpdau_completers',
              color='set_completed',
              title='Daily ARPU by completers of each event',
              width=1200,
              height=600,
              hover_data={'arpdau_completers': True, 'unique_users': True},
              category_orders={'liveops_event_definition_id': custom_order})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [112]:
fig = px.line(data_completers_all_events, 
              x='days_since_event_start', 
              y='arpdau_completers',
              color='set_completed+running',
              title='Daily ARPU by Event for completers of each event by days since event start',
              width=1200,
              height=600,
              hover_data={'arpdau_completers': True, 'unique_users': True},
              category_orders={'liveops_event_definition_id': custom_order})

#fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [113]:
fig = px.bar(data_completers_all_events_aggregated, 
              x='set_running', 
              y='arpdau',
              color='set_completed',
              barmode='group',
              title='ARPU by Completers of Each Event',
              width=1200,
              height=600,
              hover_data={'arpdau': ':.2f'},
              category_orders={'set_completed': order, 'set_running': order},
              text='arpdau')

fig.update_traces(textposition='inside', texttemplate='%{text:.2f}')

#fig.update_yaxes(tickformat='.2%', range=[max(0, min_val - padding), max_val + padding])

fig.show()

In [114]:
# hide-output
# Initialize the column if it doesn't exist
if 'arpdau_relative_diff' not in data_completers_all_events_aggregated.columns:
    data_completers_all_events_aggregated['arpdau_relative_diff'] = np.nan

# Calculate arpdau_relative_diff for all events
for event in order:
    event_data = data_completers_all_events_aggregated[
        data_completers_all_events_aggregated['set_completed'] == event
    ].copy()
    
    
    # Get the baseline value when set_running == set_completed
    baseline = event_data[event_data['set_running'] == event]['arpdau'].values
    
    if len(baseline) > 0:
        # Calculate relative diff for all rows in this event
        event_data['arpdau_relative_diff'] = (event_data['arpdau'] - baseline[0]) / baseline[0]
        
        # Update the main dataframe
        data_completers_all_events_aggregated.loc[
            data_completers_all_events_aggregated['set_completed'] == event,
            'arpdau_relative_diff'
        ] = event_data['arpdau_relative_diff'].values


data_completers_all_events_aggregated

,set_completed+running,set_completed,set_running,unique_users,iap_revenue,ad_revenue,return_rate_d1,return_rate_d7,return_rate_d14,return_rate_d28,arpdau,sequence,return_rate_d1_relative_diff,arpdau_relative_diff
0,JoesTastyTravels-JoesTastyTravels,JoesTastyTravels,JoesTastyTravels,91148,368641.479890,28880.241639,0.989848,0.984555,0.974755,0.943713,4.044428,0,0.000000,0.000000
1,JoesTastyTravels-WinterTales,JoesTastyTravels,WinterTales,85005,299422.257315,25571.936207,0.977636,0.95116,0.927703,0.891945,3.522408,1,-0.012338,-0.129071
2,JoesTastyTravels-AppletonLove,JoesTastyTravels,AppletonLove,74955,238420.051353,22797.886206,0.975290,0.951682,0.928838,0.881924,3.180843,2,-0.014707,-0.213525
3,JoesTastyTravels-Diorama,JoesTastyTravels,Diorama,65410,208236.665506,20520.856828,0.970631,0.949575,0.931155,0.901704,3.183560,3,-0.019414,-0.212853
4,JoesTastyTravels-ThroughTheAges,JoesTastyTravels,ThroughTheAges,51125,162668.367880,18018.621479,0.967707,0.948357,0.93469,0.902726,3.181777,4,-0.022368,-0.213294
5,WinterTales-JoesTastyTravels,WinterTales,JoesTastyTravels,243914,507618.062358,56102.474907,0.984291,0.978754,0.976852,0.978008,2.081135,-1,-0.005335,-0.168540
6,WinterTales-WinterTales,WinterTales,WinterTales,287236,718948.902677,73159.589295,0.989570,0.982882,0.974945,0.948222,2.502990,0,0.000000,0.000000
7,WinterTales-AppletonLove,WinterTales,AppletonLove,269756,622759.773175,68342.519251,0.981391,0.954597,0.927602,0.876751,2.308604,1,-0.008265,-0.077662
8,WinterTales-Diorama,WinterTales,Diorama,230454,495834.025716,57688.207072,0.974645,0.951947,0.930864,0.894848,2.151553,2,-0.015082,-0.140407
9,WinterTales-ThroughTheAges,WinterTales,ThroughTheAges,176399,368097.591982,48367.286245,0.971747,0.952027,0.934684,0.900845,2.086733,3,-0.018011,-0.166304


In [115]:
fig = px.line(data_completers_all_events_aggregated, 
              x='set_running', 
              y='arpdau_relative_diff',
              color='set_completed',
              title='Relative Difference in ARPU by Completers of Each Event (%)',
              width=1200,
              height=600,
              hover_data={'arpdau_relative_diff': ':.2%','unique_users': True,'iap_revenue': ':.0f'},
              category_orders={'set_completed': order, 'set_running': order},
              text='arpdau_relative_diff')

fig.update_traces(textposition='top center', texttemplate='%{text:.2%}%')
fig.update_yaxes(title='ARPDAU (%)')

fig.show()
fig = px.line(data_completers_all_events_aggregated, 
              x='sequence', 
              y='arpdau_relative_diff',
              color='set_completed',
              title='Relative Difference in ARPU by Completers of Each Event (%)',
              width=1200,
              height=600,
              hover_data={'arpdau_relative_diff': ':.2%','unique_users': True,'iap_revenue': ':.0f'},
              text='arpdau_relative_diff')

fig.update_traces(textposition='top center', texttemplate='%{text:.2%}%')
fig.update_yaxes(title='ARPDAU (%)')

fig.show()

# Prestigers

Analysis of users who reached Prestige (tier 2+) in Diorama — engagement funnel through tiers and revenue performance compared across events.

## Engagement

Distribution of Diorama users by max tier reached — overall and by day since event start — showing the funnel of how many pushed beyond tier 1.

In [116]:
# hide-output
data_diorama = data[data['liveops_event_definition_id'] == 'TimedAlbum-Diorama']

tier_reached_per_event = data_diorama[data_diorama.max_tier_reached<=10].groupby(['liveops_event_definition_id', 'max_tier_reached']).agg(
    users = ('user_id', 'nunique')
).reset_index()

# Calculate total users for each days_since_event_start
total_users_per_event = tier_reached_per_event.groupby('liveops_event_definition_id')['users'].sum().reset_index()
total_users_per_event.columns = ['liveops_event_definition_id', 'total_users_per_event']

# Merge to get total for each row
tier_reached_per_event = tier_reached_per_event.merge(total_users_per_event, on='liveops_event_definition_id', how='left')
# Calculate percentage
tier_reached_per_event['percentage_of_event_total'] = tier_reached_per_event['users'] / tier_reached_per_event['total_users_per_event']

tier_reached_per_event['max_tier_reached_str'] = tier_reached_per_event['max_tier_reached'].astype(str)
tier_reached_per_event

,liveops_event_definition_id,max_tier_reached,users,total_users_per_event,percentage_of_event_total,max_tier_reached_str
0,TimedAlbum-Diorama,1,146296,156386,0.935480,1
1,TimedAlbum-Diorama,2,8330,156386,0.053266,2
2,TimedAlbum-Diorama,3,1098,156386,0.007021,3
3,TimedAlbum-Diorama,4,383,156386,0.002449,4
4,TimedAlbum-Diorama,5,160,156386,0.001023,5
5,TimedAlbum-Diorama,6,73,156386,0.000467,6
6,TimedAlbum-Diorama,7,23,156386,0.000147,7
7,TimedAlbum-Diorama,8,13,156386,0.000083,8
8,TimedAlbum-Diorama,9,7,156386,0.000045,9
9,TimedAlbum-Diorama,10,3,156386,0.000019,10


In [117]:
fig = px.bar(tier_reached_per_event, 
              x='liveops_event_definition_id', 
              y='percentage_of_event_total',
              color='max_tier_reached_str',
              title='% users by tier reached per event',
              width=600,
              height=600,
              custom_data=['users'],
              category_orders={'max_tier_reached_str': sorted(tier_reached_per_event['max_tier_reached_str'].unique())})

fig.update_traces(hovertemplate='<b>Days Since Event Start:</b> %{x}<br><b>Max Tier Reached:</b> %{fullData.name}<br><b>Percentage:</b> %{y:.2%}<br><b>Users:</b> %{customdata[0]}<extra></extra>')
fig.update_yaxes(tickformat='.2%')
fig.update_layout(legend_title='Max Tier Reached')
fig.show()

In [118]:
# hide-output
tier_reached_diorama_daily = data_diorama[data_diorama.max_tier_reached<=10].groupby(['liveops_event_definition_id', 'days_since_event_start', 'max_tier_reached']).agg(
    users = ('user_id', 'nunique')
).reset_index()

# Calculate total users for each days_since_event_start
total_users_per_day = tier_reached_diorama_daily.groupby('days_since_event_start')['users'].sum().reset_index()
total_users_per_day.columns = ['days_since_event_start', 'total_users_per_day']

# Merge to get total for each row
tier_reached_diorama_daily = tier_reached_diorama_daily.merge(total_users_per_day, on='days_since_event_start', how='left')

# Calculate percentage
tier_reached_diorama_daily['percentage_of_day_total'] = tier_reached_diorama_daily['users'] / tier_reached_diorama_daily['total_users_per_day']

tier_reached_diorama_daily['max_tier_reached_str'] = tier_reached_diorama_daily['max_tier_reached'].astype(str)

tier_reached_diorama_daily

,liveops_event_definition_id,days_since_event_start,max_tier_reached,users,total_users_per_day,percentage_of_day_total,max_tier_reached_str
0,TimedAlbum-Diorama,0.0,1,73935,73935,1.000000,1
1,TimedAlbum-Diorama,1.0,1,87972,87972,1.000000,1
2,TimedAlbum-Diorama,2.0,1,89279,89279,1.000000,1
3,TimedAlbum-Diorama,3.0,1,89243,89243,1.000000,1
4,TimedAlbum-Diorama,4.0,1,89153,89153,1.000000,1
...,...,...,...,...,...,...,...
160,TimedAlbum-Diorama,39.0,6,57,82772,0.000689,6
161,TimedAlbum-Diorama,39.0,7,20,82772,0.000242,7
162,TimedAlbum-Diorama,39.0,8,8,82772,0.000097,8
163,TimedAlbum-Diorama,39.0,9,5,82772,0.000060,9


In [119]:
fig = px.bar(tier_reached_diorama_daily, 
              x='days_since_event_start', 
              y='percentage_of_day_total',
              color='max_tier_reached_str',
              title='% users by tier reached over days since event start',
              width=1200,
              height=600,
              custom_data=['users'],
              category_orders={'max_tier_reached_str': sorted(tier_reached_diorama_daily['max_tier_reached_str'].unique())})

fig.update_traces(hovertemplate='<b>Days Since Event Start:</b> %{x}<br><b>Max Tier Reached:</b> %{fullData.name}<br><b>Percentage:</b> %{y:.2%}<br><b>Users:</b> %{customdata[0]}<extra></extra>')
fig.update_yaxes(tickformat='.2%')
fig.update_layout(legend_title='Max Tier Reached')
fig.show()

## Revenue

ARPDAU and total IAP/ad revenue for prestige users over time and by days since event start, with a 7-day rolling average to smooth noise.

In [120]:
# hide-output
data_revenue_prestigers = data_extended[(data_extended['reached_prestige_user_id'].notna())& (data_extended['liveops_event_definition_id']!='NoTimedAlbum')].groupby(['dt', 'liveops_event_definition_id']).agg(
    unique_users=('user_id', 'nunique'),
    iap_revenue=('gross_usd_iap_revenue', 'sum'),
    ad_revenue=('gross_usd_ad_revenue', 'sum')
    ).reset_index()

data_revenue_prestigers['arpdau'] = data_revenue_prestigers['iap_revenue'] / (data_revenue_prestigers['unique_users'] * 1)  # Assuming 1 day for ARPDAU

data_revenue_prestigers

,dt,liveops_event_definition_id,unique_users,iap_revenue,ad_revenue,arpdau
0,2025-11-03,TimedAlbum-JoesTastyTravels,2880,4984.270900,582.999204,1.730650
1,2025-11-04,TimedAlbum-JoesTastyTravels,3003,5066.617876,615.339301,1.687185
2,2025-11-05,TimedAlbum-JoesTastyTravels,3010,5108.297030,600.044242,1.697109
3,2025-11-06,TimedAlbum-JoesTastyTravels,3008,4861.658616,589.537934,1.616243
4,2025-11-07,TimedAlbum-JoesTastyTravels,3023,8574.799430,599.393186,2.836520
...,...,...,...,...,...,...
201,2026-05-16,TimedAlbum-ThroughTheAges,7237,14324.036086,2012.891713,1.979278
202,2026-05-17,TimedAlbum-ThroughTheAges,7241,14346.796635,2176.536895,1.981328
203,2026-05-18,TimedAlbum-ThroughTheAges,7248,12570.457930,1936.075701,1.734335
204,2026-05-19,TimedAlbum-ThroughTheAges,7260,11274.647197,2077.871995,1.552982


In [121]:
fig = px.line(data_revenue_prestigers, 
              x='dt', 
              y='iap_revenue',
              color='liveops_event_definition_id',
              title='Daily IAP Revenue by Event',
              width=1200,
              height=600,
              hover_data={'iap_revenue': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [122]:
fig = px.line(data_revenue_prestigers, 
              x='dt', 
              y='arpdau',
              color='liveops_event_definition_id',
              title='Daily ARPU by Event',
              width=1200,
              height=600,
              hover_data={'iap_revenue': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [123]:
# hide-output
data_revenue_prestigers_dse = data_extended[(data_extended['reached_prestige_user_id'].notna())& (data_extended['liveops_event_definition_id']!='NoTimedAlbum')].groupby(['days_since_event_start', 'liveops_event_definition_id']).agg(
    unique_users=('user_id', 'nunique'),
    iap_revenue=('gross_usd_iap_revenue', 'sum'),
    ad_revenue=('gross_usd_ad_revenue', 'sum')
    ).reset_index()

data_revenue_prestigers_dse['arpdau'] = data_revenue_prestigers_dse['iap_revenue'] / (data_revenue_prestigers_dse['unique_users'] * 1)  # Assuming 1 day for ARPDAU

data_revenue_prestigers_dse

,days_since_event_start,liveops_event_definition_id,unique_users,iap_revenue,ad_revenue,arpdau
0,0.0,TimedAlbum-AppletonLove,6643,9095.161780,1500.515054,1.369135
1,0.0,TimedAlbum-Diorama,7280,17536.499946,1871.321184,2.408860
2,0.0,TimedAlbum-JoesTastyTravels,2880,4984.270900,582.999204,1.730650
3,0.0,TimedAlbum-ThroughTheAges,7711,11083.892733,1859.955301,1.437413
4,0.0,TimedAlbum-WinterTales,5951,8690.396127,1243.522725,1.460325
...,...,...,...,...,...,...
189,38.0,TimedAlbum-WinterTales,6851,10483.407751,1467.680314,1.530201
190,39.0,TimedAlbum-AppletonLove,7654,9806.158068,1693.290489,1.281181
191,39.0,TimedAlbum-Diorama,8201,19242.358822,1961.809306,2.346343
192,39.0,TimedAlbum-JoesTastyTravels,3201,12629.474148,643.597053,3.945478


In [124]:
data_revenue_prestigers_dse['arpdau_lag7'] = data_revenue_prestigers_dse.groupby('liveops_event_definition_id')['arpdau'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())

# Reshape data for plotly (separate traces for arpdau and arpdau_lag7)
fig = go.Figure()

for event in custom_order:
    if event in data_revenue_prestigers_dse['liveops_event_definition_id'].values:
        event_data = data_revenue_prestigers_dse[data_revenue_prestigers_dse['liveops_event_definition_id'] == event]
        
        fig.add_trace(go.Scatter(
            x=event_data['days_since_event_start'],
            y=event_data['arpdau'],
            mode='lines',
            name=f'{event}',
            hovertemplate='<b>Days Since Event:</b> %{x}<br><b>ARPDAU:</b> %{y:.2f}<extra></extra>'
        ))
        
        fig.add_trace(go.Scatter(
            x=event_data['days_since_event_start'],
            y=event_data['arpdau_lag7'],
            mode='lines',
            name=f'{event} (7-day avg)',
            line=dict(dash='dash'),
            hovertemplate='<b>Days Since Event:</b> %{x}<br><b>ARPDAU 7-day avg:</b> %{y:.2f}<extra></extra>'
        ))

fig.update_layout(
    title='Daily ARPU by Event with 7-day Running Average',
    width=1200,
    height=600,
    xaxis_title='Days Since Event Start',
    yaxis_title='ARPDAU'
)

fig.show()

# Economy

Analysis of currency flows (coins, energy, gems) across the full population and for completer cohorts. Focuses on whether completers are inflating gem balances relative to non-completers.

<!-- hide -->
### Prepare data

Adds a `completer_any_flag` to the economy dataframe (1 = completed any Timed Album, 0 = did not), enabling completer vs. non-completer segmentation throughout the section.

In [125]:
# hide-output
data_econ['completer_any_flag'] = data_econ['user_id'].isin(users_completers_any).astype(int)
data_econ['completer_any_flag'] = data_econ['completer_any_flag'].astype('category')
data_econ['balance_end'] = data_econ['balance_end'].astype(float)
data_econ


,dt,user_id,currency,balance_start,balance_end,n_economy_inflow,n_economy_outflow,completer_any_flag
0,2025-11-01,62893FFB7B3E6D20,energy,100.000000000,1.0,240,552,0
1,2025-11-01,BF7C8B2334C364C2,gems,380.000000000,111.0,152,421,1
2,2025-11-01,8496A0FFCD3F5280,gems,99.000000000,517.0,968,550,1
3,2025-11-01,3E9A0BF305EBDE43,coins,71.000000000,104.0,342,295,0
4,2025-11-01,A6C3D6BD9F77FC4,energy,100.000000000,0.0,140,422,0
...,...,...,...,...,...,...,...,...
56147242,2026-05-20,3837595BE4698E15,coins,1043588.000000000,1047191.0,3608,0,1
56147243,2026-05-20,24B0BFC327A93161,gems,114.000000000,114.0,31,0,0
56147244,2026-05-20,9911A346575CA63,gems,178.000000000,2.0,3,179,1
56147245,2026-05-20,5326F25E43ED0A7,coins,76.000000000,68.0,498,490,0


In [126]:
# hide-output
data_econ.dtypes

dt                      dbdate
user_id                 object
currency                object
balance_start           object
balance_end            float64
n_economy_inflow         Int64
n_economy_outflow        Int64
completer_any_flag    category
dtype: object

In [127]:
# hide-output
# using the code below create a dataframe that contails the p90, p50 and p10 for each currency by dt

econ_inflow_percentiles = data_econ.groupby(['dt','currency'])['n_economy_inflow'].quantile([0.9, 0.5, 0.1]).unstack().reset_index()
econ_outflow_percentiles = data_econ.groupby(['dt','currency'])['n_economy_outflow'].quantile([0.9, 0.5, 0.1]).unstack().reset_index()
econ_balance_end_percentiles = data_econ.groupby(['dt','currency'])['balance_end'].quantile([0.9, 0.5, 0.1]).unstack().reset_index()

# merge the inflow and outflow percentiles
econ_percentiles = pd.DataFrame()
econ_percentiles = pd.merge(econ_inflow_percentiles, econ_outflow_percentiles, on=['dt', 'currency'])
econ_percentiles = pd.merge(econ_percentiles, econ_balance_end_percentiles, on=['dt', 'currency'])

# rename dataframe columns to include the currency and the percentile
econ_percentiles.columns = ['dt', 'currency', 'p10_n_economy_inflow', 'p50_n_economy_inflow', 'p90_n_economy_inflow', 'p10_n_economy_outflow', 'p50_n_economy_outflow', 'p90_n_economy_outflow', 'p10_balance_end', 'p50_balance_end', 'p90_balance_end']

econ_energy_percentiles = econ_percentiles[econ_percentiles['currency'] == 'energy']
econ_gems_percentiles = econ_percentiles[econ_percentiles['currency'] == 'gems']
econ_coins_percentiles = econ_percentiles[econ_percentiles['currency'] == 'coins']

econ_percentiles


,dt,currency,p10_n_economy_inflow,p50_n_economy_inflow,p90_n_economy_inflow,p10_n_economy_outflow,p50_n_economy_outflow,p90_n_economy_outflow,p10_balance_end,p50_balance_end,p90_balance_end
0,2025-11-01,coins,3005.0,619.0,87.0,2120.0,250.0,0.0,147896.8,240.0,26.0
1,2025-11-01,energy,712.0,210.0,15.0,1029.0,379.0,100.0,55.0,1.0,0.0
2,2025-11-01,gems,48.0,8.0,1.0,54.0,10.0,0.0,686.0,53.0,4.0
3,2025-11-02,coins,2709.0,603.0,86.0,2020.0,260.0,0.0,141541.8,229.0,25.3
4,2025-11-02,energy,691.0,215.0,15.0,1001.0,388.0,100.0,54.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
598,2026-05-19,energy,1028.0,300.0,30.0,1408.0,526.0,120.0,68.0,1.0,0.0
599,2026-05-19,gems,61.0,9.0,1.0,70.0,10.0,0.0,1085.0,61.0,4.0
600,2026-05-20,coins,4554.9,966.0,123.0,2706.0,315.0,0.0,412081.1,385.0,42.0
601,2026-05-20,energy,1040.0,309.0,25.0,1409.0,528.0,114.0,76.0,1.0,0.0


In [128]:
# hide-output
# all users
econ_agg = data_econ.groupby(['dt','currency',]).agg(
    unique_users=('user_id', 'nunique'),
    avg_ending_balance=('balance_end', 'mean'),
    sum_inflows=('n_economy_inflow', 'sum'),
    sum_outflows=('n_economy_outflow', 'sum'),
).reset_index()

econ_agg['avg_inflows'] = econ_agg['sum_inflows'] / econ_agg['unique_users']
econ_agg['avg_outflows'] = econ_agg['sum_outflows'] / econ_agg['unique_users']

econ_agg['net_flow'] = econ_agg['avg_inflows'] - econ_agg['avg_outflows']

econ_energy_agg = econ_agg[econ_agg['currency'] == 'energy']
econ_gems_agg = econ_agg[econ_agg['currency'] == 'gems']
econ_coins_agg = econ_agg[econ_agg['currency'] == 'coins']


# completers users
econ_agg_completers = data_econ[data_econ['completer_any_flag'] == 1].groupby(['dt','currency']).agg(
    unique_users=('user_id', 'nunique'),
    avg_ending_balance=('balance_end', 'mean'),
    sum_inflows=('n_economy_inflow', 'sum'),
    sum_outflows=('n_economy_outflow', 'sum'),
).reset_index()

econ_agg_completers['avg_inflows'] = econ_agg_completers['sum_inflows'] / econ_agg_completers['unique_users']
econ_agg_completers['avg_outflows'] = econ_agg_completers['sum_outflows'] / econ_agg_completers['unique_users']

econ_agg_completers['net_flow'] = econ_agg_completers['avg_inflows'] - econ_agg_completers['avg_outflows']

econ_energy_agg_completers = econ_agg_completers[econ_agg_completers['currency'] == 'energy']
econ_gems_agg_completers = econ_agg_completers[econ_agg_completers['currency'] == 'gems']
econ_coins_agg_completers = econ_agg_completers[econ_agg_completers['currency'] == 'coins']

# non completers users
econ_agg_non_completers = data_econ[data_econ['completer_any_flag'] == 0].groupby(['dt','currency']).agg(
    unique_users=('user_id', 'nunique'),
    avg_ending_balance=('balance_end', 'mean'),
    sum_inflows=('n_economy_inflow', 'sum'),
    sum_outflows=('n_economy_outflow', 'sum'),
).reset_index()

econ_agg_non_completers['avg_inflows'] = econ_agg_non_completers['sum_inflows'] / econ_agg_non_completers['unique_users']
econ_agg_non_completers['avg_outflows'] = econ_agg_non_completers['sum_outflows'] / econ_agg_non_completers['unique_users']

econ_agg_non_completers['net_flow'] = econ_agg_non_completers['avg_inflows'] - econ_agg_non_completers['avg_outflows']

econ_energy_agg_non_completers = econ_agg_non_completers[econ_agg_non_completers['currency'] == 'energy']
econ_gems_agg_non_completers = econ_agg_non_completers[econ_agg_non_completers['currency'] == 'gems']
econ_coins_agg_non_completers = econ_agg_non_completers[econ_agg_non_completers['currency'] == 'coins']


# with completer non-completer flag
econ_agg_flagged = data_econ.groupby(['dt','currency','completer_any_flag'], observed=True).agg(
    unique_users=('user_id', 'nunique'),
    avg_ending_balance=('balance_end', 'mean'),
    sum_inflows=('n_economy_inflow', 'sum'),
    sum_outflows=('n_economy_outflow', 'sum'),
).reset_index()

econ_agg_flagged['avg_inflows'] = econ_agg_flagged['sum_inflows'] / econ_agg_flagged['unique_users']
econ_agg_flagged['avg_outflows'] = econ_agg_flagged['sum_outflows'] / econ_agg_flagged['unique_users']

econ_agg_flagged['net_flow'] = econ_agg_flagged['avg_inflows'] - econ_agg_flagged['avg_outflows']

econ_energy_agg_flagged = econ_agg_flagged[econ_agg_flagged['currency'] == 'energy']
econ_gems_agg_flagged = econ_agg_flagged[econ_agg_flagged['currency'] == 'gems']
econ_coins_agg_flagged = econ_agg_flagged[econ_agg_flagged['currency'] == 'coins']

econ_agg

,dt,currency,unique_users,avg_ending_balance,sum_inflows,sum_outflows,avg_inflows,avg_outflows,net_flow
0,2025-11-01,coins,114203,49553.098430,131621579,82757828,1152.522955,724.655464,427.86749
1,2025-11-01,energy,116070,38801.459507,33014671,56156503,284.437589,483.815827,-199.378237
2,2025-11-01,gems,81951,307.656197,2550218,2749904,31.118815,33.555466,-2.436651
3,2025-11-02,coins,118834,73830.843715,126828349,86157108,1067.273247,725.020684,342.252562
4,2025-11-02,energy,120869,37592.217707,33792854,58035860,279.582474,480.155044,-200.57257
...,...,...,...,...,...,...,...,...,...
598,2026-05-19,energy,80202,53274.240430,34531936,53701316,430.562031,669.575771,-239.01374
599,2026-05-19,gems,57275,441.749629,2175358,2199259,37.980934,38.398237,-0.417302
600,2026-05-20,coins,78735,105506.100973,140066149,89776533,1778.956614,1140.236655,638.719959
601,2026-05-20,energy,79936,53557.077187,34715675,53632745,434.293372,670.946069,-236.652697


## Gems

Deep-dive into gem currency flows — the primary currency of interest for the prestige question. Covers ending balances, net flow, inflow vs. outflow, and the completer vs. non-completer split.

### Users

Daily count of users with gem economy activity, split by completer flag — shows the relative size of the completer population in the gem economy.

In [129]:
fig = px.bar(econ_gems_agg_flagged, 
              x='dt', 
              y='unique_users',
              color='completer_any_flag',
              title='Gems unique users by day',
              width=1200,
              height=600,
              hover_data={'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

### Ending balances

Average and percentile (p10/p50/p90) gem ending balances over time for the full population, plus a distribution of total gem balance per user across the whole period.

In [130]:
fig = px.line(econ_gems_agg, 
              x='dt', 
              y='avg_ending_balance',
              #color='currency',
              title='Gems average ending balance by day',
              width=1200,
              height=600,
              hover_data={'avg_ending_balance': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(econ_gems_agg_completers, 
              x='dt', 
              y='avg_ending_balance',
              #color='currency',
              title='Gems average ending balance by day (Only-completers)',
              width=1200,
              height=600,
              hover_data={'avg_ending_balance': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(econ_gems_agg_non_completers, 
              x='dt', 
              y='avg_ending_balance',
              #color='currency',
              title='Gems average ending balance by day (Non-completers)',
              width=1200,
              height=600,
              hover_data={'avg_ending_balance': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()


### Ending Balances percentiles

Analyzing P90, P50(median) and P10 percentiles for the gems ending balance

In [131]:
# Melt the percentiles data for easier plotting
econ_gems_percentiles_melted = econ_gems_percentiles.melt(
    id_vars=['dt'],
    value_vars=['p10_balance_end', 'p50_balance_end', 'p90_balance_end'],
    var_name='percentile',
    value_name='balance'
)

fig = px.line(econ_gems_percentiles_melted, 
              x='dt', 
              y='balance',
              color='percentile',
              title='Gems ending balance percentiles by day',
              width=1200,
              height=600,
              labels={'percentile': 'Percentile', 'balance': 'Balance'},
              category_orders={'percentile': ['p10_balance_end', 'p50_balance_end', 'p90_balance_end']})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [132]:
# hide-output
distribution_date = pd.DataFrame()
distribution_data = data_econ.groupby(['user_id','currency']).agg(
    total_inflow=('n_economy_inflow', 'sum'),
    total_outflow=('n_economy_outflow', 'sum'),
    balance_end=('balance_end', 'mean'),
).reset_index()

distribution_data_gems = distribution_data[distribution_data['currency'] == 'gems']

distribution_data

,user_id,currency,total_inflow,total_outflow,balance_end
0,100006BF6559BF99,coins,252,224,28.000000
1,100006BF6559BF99,energy,397,505,0.000000
2,100006BF6559BF99,gems,8,30,28.000000
3,10003F6FA8F5957E,coins,65071,66129,109.024096
4,10003F6FA8F5957E,energy,27876,61770,5.413174
...,...,...,...,...,...
1937540,FFFFF4ACE971353A,coins,3470,3490,121.857143
1937541,FFFFF4ACE971353A,energy,2557,3927,28.571429
1937542,FFFFF4ACE971353A,gems,77,70,16.666667
1937543,FFFFF67B4ED7017A,coins,9,5,4.000000


In [133]:
# hide-output
# Pre-aggregate gems data by balance_end ranges
gems_data = distribution_data_gems[distribution_data_gems['balance_end']<=1000].copy()
gems_data['balance_end'] = pd.to_numeric(gems_data['balance_end'], errors='coerce')

# Create histogram data with 100 bins
gems_hist, bin_edges = np.histogram(gems_data['balance_end'].dropna(), bins=500)

# Create aggregated dataframe
gems_agg = pd.DataFrame({
    'balance_end': (bin_edges[:-1] + bin_edges[1:]) / 2,  # bin centers
    'count': gems_hist
})

# Calculate statistics
p10 = gems_data['balance_end'].quantile(0.1)
p50 = gems_data['balance_end'].quantile(0.5)
p90 = gems_data['balance_end'].quantile(0.9)
mean_balance = gems_data['balance_end'].mean()

# Create histogram
fig = px.bar(
    gems_agg,
    x='balance_end',
    y='count',
    title='Gems Ending Balance Distribution (with P10, P50, P90, and Mean)',
    labels={'balance_end': 'Ending Balance', 'count': 'Number of Users'},
    width=1200,
    height=600,
)

# Add vertical lines for percentiles and mean
fig.add_vline(x=p10, line_dash='dash', line_color='blue', annotation_text='P10', annotation_position='top')
fig.add_vline(x=p50, line_dash='dash', line_color='green', annotation_text='P50 (Median)', annotation_position='top')
fig.add_vline(x=p90, line_dash='dash', line_color='red', annotation_text='P90', annotation_position='top')
fig.add_vline(x=mean_balance, line_dash='dot', line_color='purple', annotation_text='Mean', annotation_position='top')

fig.update_layout(showlegend=False)

fig.show()

### Net flow

Daily net gem flow (inflows − outflows) for the full population — positive values indicate net gem accumulation, negative indicate net spending.

In [134]:
fig = px.line(econ_gems_agg, 
              x='dt', 
              y='net_flow',
              #color='currency',
              title='Gems net flow by day',
              width=1200,
              height=600,
              hover_data={'net_flow': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

# add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="gray") 

fig.show()

fig = px.line(econ_gems_agg_completers, 
              x='dt', 
              y='net_flow',
              #color='currency',
              title='Gems net flow by day (Only-Completers)',
              width=1200,
              height=600,
              hover_data={'net_flow': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

# add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="gray") 

fig.show()

fig = px.line(econ_gems_agg_non_completers, 
              x='dt', 
              y='net_flow',
              #color='currency',
              title='Gems net flow by day (Non-completers)',
              width=1200,
              height=600,
              hover_data={'net_flow': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

# add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="gray") 

fig.show()

### Inflow vs Outflow

Average gem inflows and outflows plotted side-by-side to show whether activity is driven by earning or spending, and how the two move together over time.

In [135]:
# Melt the dataframe to have inflows and outflows as separate rows
econ_gems_melted = econ_gems_agg.melt(
    id_vars=['dt', 'unique_users'], 
    value_vars=['avg_inflows', 'avg_outflows'],
    var_name='flow_type',
    value_name='amount'
)

fig = px.line(econ_gems_melted, 
              x='dt', 
              y='amount',
              color='flow_type',
              title='Gems average inflows and outflows by day',
              width=1200,
              height=600,
              hover_data={'amount': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

# Melt the dataframe to have inflows and outflows as separate rows
econ_gems_melted_completers = econ_gems_agg_completers.melt(
    id_vars=['dt', 'unique_users'], 
    value_vars=['avg_inflows', 'avg_outflows'],
    var_name='flow_type',
    value_name='amount'
)

fig = px.line(econ_gems_melted_completers, 
              x='dt', 
              y='amount',
              color='flow_type',
              title='Gems average inflows and outflows by day (Only-Completers)',
              width=1200,
              height=600,
              hover_data={'amount': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

# Melt the dataframe to have inflows and outflows as separate rows
econ_gems_melted_non_completers = econ_gems_agg_non_completers.melt(
    id_vars=['dt', 'unique_users'], 
    value_vars=['avg_inflows', 'avg_outflows'],
    var_name='flow_type',
    value_name='amount'
)

fig = px.line(econ_gems_melted_non_completers, 
              x='dt', 
              y='amount',
              color='flow_type',
              title='Gems average inflows and outflows by day (Non-Completers)',
              width=1200,
              height=600,
              hover_data={'amount': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

<!-- hide --> 
# Economy by completer cohort (work in progress)

Breaks down gem economy metrics (ending balance, inflows, outflows) separately for each completer cohort, allowing comparison of gem behaviour across events.

<!-- hide --> 
## Completers for each event

Tracks average gem ending balance, inflows, and outflows over time for each completer cohort (JoesTastyTravels, WinterTales, AppletonLove, Diorama), and compares completers vs. non-completers.

In [141]:
# hide-output
# Create a loop to iterate through all events and aggregate results
events = custom_order
user_sets = [users_completer_JoesTastyTravels, users_completer_WinterTales, users_completer_AppletonLove, users_completer_Diorama, users_completer_ThroughTheAges]
#user_sets = [users_near_completer_JoesTastyTravels, users_near_completer_WinterTales, users_near_completer_AppletonLove, users_near_completer_Diorama, users_near_completer_ThroughTheAges]
user_set_names = ['JoesTastyTravels', 'WinterTales', 'AppletonLove', 'Diorama', 'ThroughTheAges']
filter_mask = ['JoesTastyTravels', 'WinterTales', 'AppletonLove', 'Diorama','ThroughTheAges']
results_completers_list = []
results_non_completers_list = []

for set, set_name in zip(user_sets, user_set_names):
    econ_completers_data = data_econ[(data_econ['user_id'].isin(set))]

    econ_completers_aggregated = econ_completers_data.groupby(['dt','currency']).agg(
        unique_users=('user_id', 'nunique'),
        avg_ending_balance=('balance_end', 'mean'),
        avg_inflows=('n_economy_inflow', 'mean'),
        avg_outflows=('n_economy_outflow', 'mean'),
    ).reset_index()

    econ_completers_aggregated['set_completed'] = set_name

    econ_non_completers_data = data_econ[~data_econ['user_id'].isin(set)]

    econ_non_completers_aggregated = econ_non_completers_data.groupby(['dt','currency']).agg(
        unique_users=('user_id', 'nunique'),
        avg_ending_balance=('balance_end', 'mean'),
        avg_inflows=('n_economy_inflow', 'mean'),
        avg_outflows=('n_economy_outflow', 'mean'),
    ).reset_index()

    econ_non_completers_aggregated['set_completed'] = set_name

    results_completers_list.append(econ_completers_aggregated)
    results_non_completers_list.append(econ_non_completers_aggregated)

econ_completers_all_events = pd.concat(results_completers_list, ignore_index=True)
econ_non_completers_all_events = pd.concat(results_non_completers_list, ignore_index=True)

econ_completers_all_events = econ_completers_all_events[econ_completers_all_events['set_completed'].isin(filter_mask)]
econ_non_completers_all_events = econ_non_completers_all_events[econ_non_completers_all_events['set_completed'].isin(filter_mask)]

econ_completers_all_events

,dt,currency,unique_users,avg_ending_balance,avg_inflows,avg_outflows,set_completed
0,2025-11-01,coins,2094,235751.076886,3403.244625,1285.267431,JoesTastyTravels
1,2025-11-01,energy,2098,36.131554,689.850602,1018.352717,JoesTastyTravels
2,2025-11-01,gems,1953,891.100358,252.472122,184.845878,JoesTastyTravels
3,2025-11-02,coins,2113,237235.461903,2953.768102,1135.936583,JoesTastyTravels
4,2025-11-02,energy,2116,31.526465,625.657279,973.316163,JoesTastyTravels
...,...,...,...,...,...,...,...
3010,2026-05-19,energy,2223,217.201529,1388.248534,1784.834908,ThroughTheAges
3011,2026-05-19,gems,2208,2104.759964,451.391407,352.290308,ThroughTheAges
3012,2026-05-20,coins,2208,691694.168025,6779.189312,1638.984149,ThroughTheAges
3013,2026-05-20,energy,2211,242.617820,1414.958163,1748.354138,ThroughTheAges


In [142]:
# hide-output
econ_completers_energy_agg = econ_completers_all_events[econ_completers_all_events['currency'] == 'energy']
econ_completers_gems_agg = econ_completers_all_events[econ_completers_all_events['currency'] == 'gems']
econ_completers_coins_agg = econ_completers_all_events[econ_completers_all_events['currency'] == 'coins']

econ_noncompleters_energy_agg = econ_non_completers_all_events[econ_non_completers_all_events['currency'] == 'energy']
econ_noncompleters_gems_agg = econ_non_completers_all_events[econ_non_completers_all_events['currency'] == 'gems']
econ_noncompleters_coins_agg = econ_non_completers_all_events[econ_non_completers_all_events['currency'] == 'coins']



econ_completers_gems_agg

,dt,currency,unique_users,avg_ending_balance,avg_inflows,avg_outflows,set_completed
2,2025-11-01,gems,1953,891.100358,252.472122,184.845878,JoesTastyTravels
5,2025-11-02,gems,1971,862.493151,208.212099,154.093354,JoesTastyTravels
8,2025-11-03,gems,2068,885.570116,245.655556,189.468569,JoesTastyTravels
11,2025-11-04,gems,2069,875.826970,223.474845,192.301595,JoesTastyTravels
14,2025-11-05,gems,2087,886.870149,186.665641,163.252995,JoesTastyTravels
...,...,...,...,...,...,...,...
3002,2026-05-16,gems,2163,1587.327323,560.865085,321.655109,ThroughTheAges
3005,2026-05-17,gems,2168,1744.440498,518.909436,322.159594,ThroughTheAges
3008,2026-05-18,gems,2211,2025.799638,667.751057,345.680235,ThroughTheAges
3011,2026-05-19,gems,2208,2104.759964,451.391407,352.290308,ThroughTheAges


In [143]:
fig = px.line(econ_completers_gems_agg, 
              x='dt', 
              y='avg_ending_balance',
              color='set_completed',
              title='Gems average ending balance by day',
              width=1200,
              height=600,
              hover_data={'avg_ending_balance': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

fig = px.line(econ_noncompleters_gems_agg, 
              x='dt', 
              y='avg_ending_balance',
              color='set_completed',
              title='Gems average ending balance by day',
              width=1200,
              height=600,
              hover_data={'avg_ending_balance': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)

fig.show()

In [144]:
# Add set to legends for completers vs non-completers gems comparison
# Completers
econ_completers_gems_melted = econ_completers_gems_agg.melt(
    id_vars=['dt', 'set_completed', 'unique_users'], 
    value_vars=['avg_inflows', 'avg_outflows'],
    var_name='flow_type',
    value_name='amount'
)

fig = px.line(econ_completers_gems_melted, 
              x='dt', 
              y='amount',
              color='flow_type',
              facet_row='set_completed',
              title='Gems average inflows and outflows by day - Completers',
              width=1200,
              height=1200,
              hover_data={'amount': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)
fig.show()

# Non-completers
econ_noncompleters_gems_melted = econ_noncompleters_gems_agg.melt(
    id_vars=['dt', 'set_completed', 'unique_users'], 
    value_vars=['avg_inflows', 'avg_outflows'],
    var_name='flow_type',
    value_name='amount'
)

fig = px.line(econ_noncompleters_gems_melted, 
              x='dt', 
              y='amount',
              color='flow_type',
              facet_row='set_completed',
              title='Gems average inflows and outflows by day - Non-Completers',
              width=1200,
              height=1200,
              hover_data={'amount': True, 'unique_users': True})

fig = add_vlines_to_figure(fig, vlines_events)
fig.show()

<!-- hide --> 
## Suggested additional analysis

The following analyses would help answer questions not yet fully covered in the notebook.

### 1. Product category breakdown for completers
The IAP revenue share of completers is clear (15.4%), but we don't know *what* they're buying. Adding a product category dimension to the IAP revenue query (e.g. gems, energy bundles, sticker packs, passes) would reveal whether completers are buying to complete sets or spending on other things — key for understanding their monetisation driver.

### 2. Prestige cadence — when do users prestige relative to event start?
The tier-by-day chart shows the distribution of tiers over time, but doesn't show *when* users first crossed into prestige. A survival curve of time-to-prestige (days since event start) would reveal whether prestige is being hit early (driven by whales) or late (driven by casual grinders who get there by the end).

### 3. Repeat prestige behaviour across events (ThroughTheAges)
ThroughTheAges is the second event with prestige. Compare the prestige rate and speed in ThroughTheAges vs. Diorama — do users who prestiged in Diorama prestige again faster? This would show whether prestige creates habitual behaviour and informs whether it is re-engaging the same cohort or acquiring new ones.

### 4. Hangover duration — when does completer activity normalise?
The hangover section tracks return rate and revenue, but doesn't surface a clear answer to "how many days after an event does the completer bump last?" Fitting a decay curve to post-event ARPDAU or plotting the delta between completers and non-completers by days-after-event would give an answer.

In [145]:
export_notebook_html(
    notebook_path='./prestige.ipynb',
    output_path='./prestige.html',
)

Saved to prestige.html


PosixPath('prestige.html')